# Colab 訓練啟動器

執行 `aicup.train_rank`，checkpoint 存到 `rank-trained-models/<commit hash>/`，結束後畫出每個 epoch 的數值。

1. 執行階段選 GPU（smoke test用 T4，正式訓練用 A100）
2. 改第一格的 `MODE` 等設定
3. 執行階段 → 全部執行

smoke test讀 `rank-features-10pct/`，Drive 上要先有這個資料夾。全量特徵載入時要約 22 GB 系統 RAM，標準 T4 放不下。

In [ ]:
MODE = 'smoke'          # 'smoke' smoke test，'full' 正式訓練
EXTRA_ARGS = []         # 這次改動要加的參數，例如 ['--epochs', '2']
COMPARE = []            # 一起畫圖比較的目錄名，例如 ['a1b2c3d']
AUTO_UNASSIGN = True    # 預設跑完自動釋放 runtime

FEAT_DIR = 'rank-features-10pct' if MODE == 'smoke' else 'rank-features'
MIN_RAM_GB = 4 if MODE == 'smoke' else 30   # 載入打包特徵所需的系統 RAM

In [ ]:
import os, subprocess, sys
from google.colab import drive, userdata

drive.mount('/content/drive')
REPO = '/content/aicup-2026-go'
DRIVE_ROOT = '/content/drive/MyDrive/aicup-2026'


def git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode:
        print(r.stderr.replace(userdata.get('GH_TOKEN'), '***'))
        raise SystemExit('git %s 失敗' % args[0])
    return r.stdout.strip()


if not os.path.exists(REPO):
    git('clone', '-q', 'https://%s@github.com/Jx-study/aicup-2026-go.git' % userdata.get('GH_TOKEN'), REPO)
os.chdir(REPO)
git('pull', '-q')
print(git('log', '-1', '--oneline'))

# 子進程的輸出要逐行轉印，否則 notebook 看不到；!python 失敗時 notebook 會繼續往下跑
p = subprocess.Popen([sys.executable, '-u', '-m', 'aicup.colab_setup'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
if p.wait():
    raise SystemExit('colab_setup 失敗，Drive 斷線時重開 runtime 再試')

# colab_setup 只連 rank-features，smoke test的特徵目錄另外連
if FEAT_DIR != 'rank-features':
    target = os.path.join(DRIVE_ROOT, FEAT_DIR)
    if not os.path.isdir(target):
        raise SystemExit('Drive 上沒有 %s' % target)
    if not os.path.islink(FEAT_DIR):
        os.symlink(target, FEAT_DIR)

ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024 ** 3
print('系統 RAM %.1f GB，%s 需要 %d GB' % (ram_gb, FEAT_DIR, MIN_RAM_GB))
if ram_gb < MIN_RAM_GB:
    raise SystemExit('RAM 不足，載入特徵時 kernel 會被 kill')
!nproc
!nvidia-smi

In [ ]:
HASH = git('rev-parse', '--short', 'HEAD')
if MODE == 'smoke':
    CKPT_DIR = 'rank-trained-models/smoke/' + HASH
    train_args = ['--epochs', '1', '--max-steps', '200', '--fresh']
else:
    CKPT_DIR = 'rank-trained-models/' + HASH
    train_args = []
train_args += ['--feat-dir', FEAT_DIR, '--ckpt-dir', CKPT_DIR, *EXTRA_ARGS]
os.makedirs(CKPT_DIR, exist_ok=True)
LOG = os.path.join(CKPT_DIR, 'train.log')
print('mode', MODE, '| ckpt', CKPT_DIR, '| args', train_args)

# 輸出同時寫到 Drive，runtime 釋放後仍能看錯誤訊息
with open(LOG, 'a') as log:
    p = subprocess.Popen([sys.executable, '-u', '-m', 'aicup.train_rank', *train_args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
TRAIN_RC = p.wait()
print('train_rank 結束碼', TRAIN_RC)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 看舊結果時把 CKPT_DIR 改成 'rank-trained-models/<hash>'
runs = {}
for d in [CKPT_DIR, *('rank-trained-models/' + c for c in COMPARE)]:
    path = os.path.join(d, 'history.csv')
    if os.path.exists(path):
        runs[os.path.basename(d)] = pd.read_csv(path)
    else:
        print('找不到', path)

if runs:
    panels = [('Loss', [('train_loss', '-'), ('val_loss', '--')]),
              ('Accuracy (%)', [('train_acc', '-'), ('val_acc', '--')]),
              ('Val score', [('val_score_pick', '-'), ('val_score_argmax', '--')]),
              ('Minutes per epoch', [('total_minutes', '-')])]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, (title, cols) in zip(axes.flat, panels):
        for i, (name, h) in enumerate(runs.items()):
            for col, style in cols:
                ax.plot(h['epoch'], h[col], style, color='C%d' % i, marker='o', label='%s %s' % (name, col))
        ax.set_title(title)
        ax.set_xlabel('epoch')
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(CKPT_DIR, 'history.png'), dpi=120)
    plt.show()
    for name, h in runs.items():
        print(name)
        display(h.round(4))

In [ ]:
from google.colab import runtime

drive.flush_and_unmount()   # 等 Drive 寫入完成
if AUTO_UNASSIGN:
    runtime.unassign()
else:
    print('未釋放 runtime，用完記得手動中斷')